In [15]:
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

model = YOLO("yolov8n.pt")

video_path = "/content/gate.webm"  # IMPORTANT: Colab path
cap = cv2.VideoCapture(video_path)

line_y = 250  #  adjust this based on your video

entry_count = 0
exit_count = 0

prev_centers = []
counted = set()  # prevent double counting

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame , verbose=False)

    current_centers = []

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            label = model.names[cls]

            if label == "car":
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                current_centers.append((cx, cy))

                cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
                cv2.circle(frame, (cx, cy), 5, (0,0,255), -1)

    # Compare positions
    for i, (cx, cy) in enumerate(current_centers):
        for j, (px, py) in enumerate(prev_centers):

            distance = ((cx - px)**2 + (cy - py)**2)**0.5

            if distance < 80:  # 👈 increased threshold

                obj_id = (j)  # simple ID

                if obj_id not in counted:

                    # ENTRY
                    if py < line_y and cy >= line_y:
                        entry_count += 1
                        counted.add(obj_id)

                    # EXIT
                    elif py > line_y and cy <= line_y:
                        exit_count += 1
                        counted.add(obj_id)

    prev_centers = current_centers.copy()

    cv2.line(frame, (0, line_y), (frame.shape[1], line_y), (255,0,0), 2)

    cv2_imshow(frame)

cap.release()
cv2.destroyAllWindows()

print("Entries:", entry_count)
print("Exits:", exit_count)

Output hidden; open in https://colab.research.google.com to view.